# Evaluate any classifier · optional template

**Bring predictions—not a new model.** This notebook expects one row
per evaluated case and keeps the scientific checks visible.

Required columns: **true_class**, **prediction**, **group**, and
**date**. A group is a real unit that must not be split across train
and test.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, f1_score

root = next((p for p in [Path(".."), Path(".")] if (p / "data").exists()), None)
raw = "https://raw.githubusercontent.com/Facuriy/agro-data-analysis-hands-on/v1.0.0"

# Demo uses the released real teaching predictions.
source = root / "data" / "cnn_model_outputs.csv" if root else f"{raw}/data/cnn_model_outputs.csv"
predictions = pd.read_csv(source).rename(
    columns={
        "cnn_prediction": "prediction",
        "held_out_block": "group",
    }
)
predictions.head()

## 1 · Audit before metrics

Replace the demo table only after your own export contains the same
four fields. Do not invent groups.

In [ ]:
required = {"true_class", "prediction", "group", "date"}
assert required.issubset(predictions.columns), required - set(predictions.columns)
assert predictions[list(required)].notna().all().all()
assert len(predictions) == len(predictions.drop_duplicates())

audit = pd.DataFrame({
    "rows": [len(predictions)],
    "groups": [predictions.group.nunique()],
    "dates": [predictions.date.nunique()],
    "classes": [predictions.true_class.nunique()],
})
display(audit)

## 2 · Evaluate by the grouping-relevant context

In [ ]:
rows = []
for date, data in predictions.groupby("date"):
    rows.append({
        "date": date,
        "n": len(data),
        "correct": int((data.true_class == data.prediction).sum()),
        "accuracy": accuracy_score(data.true_class, data.prediction),
        "macro_F1": f1_score(
            data.true_class, data.prediction,
            average="macro", zero_division=0,
        ),
    })
metrics = pd.DataFrame(rows)
display(metrics.round(3))

labels = sorted(predictions.true_class.unique())
fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4))
axes = [axes] if len(metrics) == 1 else axes
for ax, (date, data) in zip(axes, predictions.groupby("date")):
    ConfusionMatrixDisplay.from_predictions(
        data.true_class, data.prediction,
        labels=labels, colorbar=False, cmap="Blues", ax=ax,
    )
    ax.set_title(str(date))
plt.tight_layout()
plt.show()

## 3 · Vibe coding with verification

Ask an AI assistant:

> Write pandas/scikit-learn code that reproduces the metrics table
> by date. Do not modify predictions. Return one tidy table.

Before accepting its output, verify:

- row count and groups are unchanged;
- confusion-matrix cells sum to n;
- correct counts equal the diagonal sums;
- no claim treats repeated dates as independent new field units.

**Results sentence:**  
“Under [grouped split], the model correctly classified __/__ cases
in [context]; the main confusion was __.”

**Discussion sentence:**  
“This evaluation does not establish __; the next independent test
should include __.”

Sources:
[GroupKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) ·
[model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html) ·
[structured cross-validation](https://doi.org/10.1111/ecog.02881)